<table style="width:100%; border-bottom: 2px solid #ccc; margin-bottom: 20px;">
  <tr>
    <td style="vertical-align:middle;">
      <img src="../../resources/ADI-Logo-RGB-FullColor.png" alt="Company Logo" height="30">
    </td>
    <td style="text-align:right; vertical-align:middle;">
      <p style="margin: 0;">Phased Array Systems</p>
      <p style="font-size: 14px; margin: 0;">Iain Derrington – ADEF Group, ADI</p>
      <p style="font-size: 12px; color: #555;">Field Applications & Platform Engineer</p>
    </td>
  </tr>
</table>

In [ ]:
# Common Declarations and setup

import os
import sys
sys.path.insert(0, '../src')
import time
import asyncio
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg
import matplotlib.gridspec as gridspec

from io import BytesIO

%matplotlib widget

from pathlib import Path
from phaser_functions import *
from phaser_init import init_phaser_sdr

from adi import adf4159
from adi import ad9361
from adi import one_bit_adc_dac
from adi import ad9361
from adi import tddn
from adi.cn0566 import CN0566

import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output, HTML 

from dataclasses import dataclass, fields
from typing import List



# Get Script / Notebook root and full path to resources folder
phaser_root = get_phaser_root()
resource_path = phaser_root / "resources"

#display(Markdown(f"Phaser root: **{phaser_root}**"))
#display(Markdown(f"Resource path: **{resource_path}**\n"))

In [ ]:
# Configuration class

@dataclass
class RadarConfig:
    """Configuration parameters for FMCW radar operation."""

    # ========== SDR Parameters ==========
    sample_rate: float = 5.85e6  # NOTE: This is a placeholder! The actual sample rate is calculated
                                  # in configure_sdr() based on buffer_size / frame_time to ensure
                                  # we capture exactly one complete frame (chirp + padding).
                                  # Formula: sample_rate = sdr_buf_size / (ramp_time + pri_padding_ms)
                                  # For 4096 samples / 0.7ms = 5.85 MHz
                                  # ALWAYS use sdr.sample_rate (actual) not config.sample_rate (default) in plots!

    center_freq: float = 2.1e9  # SDR LO frequency (Hz). Upconverted to output_freq by ADF4159
                                 # Keep at 2.1 GHz for optimal Pluto performance

    signal_freq: float = 100e3  # TX baseband tone frequency (Hz). Creates IF offset
                                 # Used to separate DC offset from target returns
                                 # Typical: 100 kHz. Don't change unless you know why

    rx_gain: int = 30  # Receiver gain (dB). Range: -3 to 70 dB
                        # Higher = more sensitive but risk ADC saturation on strong returns
                        # Start at 60, reduce if seeing saturation artifacts
                        # Trade-off: +10 dB gain ~= 3x detection range OR 10 dB less TX power needed

    tx_gain: int = 0  # Transmitter gain (dB). Range: 0 to -88 dB (0 = max power)
                       # 0 dB ~= +30 dBm EIRP with array gain ~= 1W effective
                       # Reduce for short range, regulations, or power saving
                       # Trade-off: -6 dB power ~= 0.5x detection range

    sdr_buf_size: int = 1024 * 8
    fft_size: int = sdr_buf_size

    # ========== Chirp Parameters ==========
    output_freq: float = 9.9e9  # Radar transmit frequency (Hz). X-band (8-12 GHz)
                                 # 9.9 GHz = 30mm wavelength. Good for small targets
                                 # Check local regulations (ISM, amateur, Part 15)

    chirp_BW: float = 500e6  # Chirp bandwidth (Hz). Determines range resolution
                              # Range resolution = c/(2*BW) = 0.3m at 500 MHz
                              # Wider BW = better resolution, more processing
                              # Typical: 250 MHz to 1 GHz (if hardware supports)
                              # Trade-off: 2x BW = 0.5x range resolution (better)

    ramp_time_us: int = 500  # Chirp duration (microseconds). Affects max range
                           # Longer = more samples per chirp = better range resolution
                           # Also affects PRF (pulse repetition frequency)
                           # Typical: 100 to 1000 us
                           # Trade-off: 2x ramp_time_us = 0.5x PRF = 0.5x max unambiguous velocity

    num_chirps: int = 128  # Number of chirps per frame (CPI - Coherent Processing Interval)
                            # More chirps = better Doppler (velocity) resolution
                            # Doppler resolution = lambda/(2*CPI*PRI) where PRI ~= ramp_time_us
                            # Typical: 64 to 512. Power of 2 for efficient FFT
    

    # ========== Array Parameters ==========
    element_spacing: float = 0.014  # Antenna element spacing (meters). 14mm ~= lambda/2 at 10 GHz
                                     # lambda/2 spacing prevents grating lobes (spatial aliasing)
                                     # Don't change unless physical array changes

    gain_list: List[int] = None  # Per-element gain (0-127). None = all max (127)
                                  # Can apply taper (Blackman, Taylor) to reduce sidelobes
                                  # Example: [8, 34, 84, 127, 127, 84, 34, 8] for Blackman
                                  # Trade-off: Tapering reduces sidelobes but lowers gain

    # ========== Timing Parameters (Advanced) ==========
    begin_offset_fraction: float = 0.05  # Fraction of chirp to skip at start (0.0-0.3)
                                         # VCO takes time to settle; early samples are non-linear
                                         # 0.1 = skip first 10% of chirp (30 us at 300 us ramp)
                                         # Increase if seeing range artifacts near zero
                                         # Trade-off: More offset = fewer samples = less SNR

    pri_padding_ms: float = 0.1   # Dead time between chirps (milliseconds)
                                  # Allows VCO to reset and prevents chirp overlap
                                  # PRI (Pulse Repetition Interval) = ramp_time_us + padding
                                  # Affects PRF and max unambiguous velocity
                                  # Trade-off: More padding = lower PRF = lower max velocity

    # ========== TDD (Time Division Duplex) Parameters ==========
    tdd_trigger_on_raw: int = 0   # TDD GPIO trigger start (raw units)
                                   # Synchronizes chirp generation with data capture
                                   # Keep at 0 for immediate trigger

    tdd_trigger_off_raw: int = 20  # TDD GPIO trigger stop (raw units)
                                    # Pulse width for trigger signal
                                    # Typical: 5-20. Must be long enough for hardware to latch
                                    # Trade-off: Longer pulse more reliable but delays start

    rpi_ip:       str = "192.168.1.10"
    #rpi_ip:       str = "phaser.local"
    sdr_ip:       str = "192.168.2.1"
    fieldfox_ip:  str = "192.168.1.30"

    def __post_init__(self):
        """Set default gain list and calibration file paths if not provided."""
        if self.gain_list is None:
            self.gain_list = [127] * 8

    def __iter__(self):
        for field in fields(self):
            yield field.name, getattr(self, field.name)

#  Default are stored in a dataclass
config = RadarConfig()

#display(Markdown("#### Config values"))
#for name, value in config:    
#    display(Markdown(f"{name} = {value}"))

pll    = None      
gpio   = None
tdd    = None
phaser = None
sdr    = None


In [ ]:
# Helper Functions 
def connect_devices():
    md = """ """
    try:
        global pll, gpio, tdd, phaser, sdr
        display(Markdown(f"- ADF4159: ip: {config.rpi_ip}"))
        pll    = adf4159        (uri="ip:" + config.rpi_ip)
        
        display(Markdown(f"- GPIO: ip: {config.sdr_ip}"))
        gpio   = one_bit_adc_dac(uri="ip:" + config.sdr_ip)

        display(Markdown(f"- TDDN: ip: {config.sdr_ip}"))
        tdd = tddn(uri="ip:" + config.sdr_ip)

        display(Markdown(f"- Phaser: ip: {config.rpi_ip}"))
        phaser = CN0566         (uri="ip:" + config.rpi_ip)

        display(Markdown(f"- SDR: ip: {config.sdr_ip}"))
        sdr    = ad9361         (uri="ip:" + config.sdr_ip)
    
    except:
        display(Markdown(f"Unable to connect to CN0566 / ADF4159. Please check the IP addresses and connections."))
        print("")
        sys.exit(1)

    phaser.sdr = sdr
    
def configure_phaser():
    phaser.configure(device_mode="rx")
    phaser.element_spacing = config.element_spacing
    
    for i in range(0, 8):
        phaser.set_chan_phase(i, 0)

    display(Markdown(f"- Phaser channel phase  = 0"))
    
    for i in range(0, len(config.gain_list)):
        phaser.set_chan_gain(i, config.gain_list[i], apply_cal=False)
    
    phaser._gpios.gpio_tx_sw = 0
    phaser._gpios.gpio_vctrl_1 = 1
    phaser._gpios.gpio_vctrl_2 = 1

def configure_sdr():   
    destroy_sdr_buffer()
    
    # Configure sample rate to capture the full frame (chirp + padding) x no of chirps
    # 
    frame_time_s = ( (config.ramp_time_us * 1e-6) + (config.pri_padding_ms * 1e-3) ) * config.num_chirps

    sr = int(config.sdr_buf_size / frame_time_s)
    phaser.sdr.sample_rate = sr
    
    phaser.sdr.rx_lo = int(config.center_freq)
    phaser.sdr.rx_enabled_channels = [0, 1]
    phaser.sdr.rx_buffer_size = config.sdr_buf_size
    
    phaser.sdr.gain_control_mode_chan0 = 'manual'
    phaser.sdr.gain_control_mode_chan1 = 'manual'
    phaser.sdr.rx_hardwaregain_chan0 = config.rx_gain
    phaser.sdr.rx_hardwaregain_chan1 = config.rx_gain

    phaser.sdr.tx_buffer_size = config.sdr_buf_size
    phaser.sdr.tx_lo = int(config.center_freq)
    phaser.sdr.tx_enabled_channels = [0, 1]
    phaser.sdr.tx_cyclic_buffer = True
    phaser.sdr.tx_hardwaregain_chan0 = -88
    phaser.sdr.tx_hardwaregain_chan1 = int(config.tx_gain)

    display(Markdown(f"""- Setting sample  rate: {sr/1e6:.2f} MHz
- Actual Sample rate: {sdr.sample_rate/1e6:.2f} MHz
- Frame time: {frame_time_s*1e3:.2f} ms (chirp + padding) * Num Chirps
- Chirp time: {config.ramp_time_us} us
- Padding time: {config.pri_padding_ms} ms
- RX LO: {sdr.rx_lo/1e9:.1f} GHz
- TX LO: {sdr.tx_lo/1e9:.1f} GHz
- Buffer size: {sdr.rx_buffer_size} samples
- Capture time: {sdr.rx_buffer_size/sdr.sample_rate*1e3:.2f} ms"""))

def destroy_sdr_buffer():
    try: sdr.tx_destroy_buffer()
    except: pass
    try: sdr.rx_destroy_buffer()
    except: pass

def configure_adf4159():
    # Configure ADF4159 for triggered sawtooth chirp

    vco_freq = int(config.output_freq + config.signal_freq + config.center_freq)
    BW = config.chirp_BW
    num_steps = int(config.ramp_time_us)

    phaser.frequency = int(vco_freq / 4)
    phaser.freq_dev_range = int(BW / 4)
    phaser.freq_dev_step = int((BW / 4) / num_steps)
    phaser.freq_dev_time = int(config.ramp_time_us)
    
    phaser.delay_word = 4095
    phaser.delay_clk = "PFD"
    phaser.delay_start_en = 0
    phaser.ramp_delay_en = 0
    phaser.trig_delay_en = 0
    phaser.ramp_mode = "single_sawtooth_burst"
    phaser.sing_ful_tri = 0
    phaser.tx_trig_en = 1
    phaser.enable = 0                                  # 0 = PLL enable.  Write this last to update all the registers

    display(Markdown(f"""- VCO Output Frequency = {vco_freq/1e9} GHz
- Chirp BW: {(4 * phaser.freq_dev_range)/1e6:.0f} MHz
- Ramp time: {config.ramp_time_us} us
- Chirp rate: {config.chirp_BW/(config.ramp_time_us*1e-6)/1e12:.2f} THz/s"""))

def configure_tdd():
    """
    Configure the TDD (Time Division Duplex) controller so that each FMCW chirp
    is synchronised to the data acquisition hardware.

    The TDD engine generates trigger signals at the start of each chirp period
    (PRI) and repeats this for the required number of chirps in a burst.
    """

    # Route the TDD trigger to the external sync circuitry and enable
    # the Phaser board trigger path.
    gpio.gpio_tdd_ext_sync = True
    gpio.gpio_phaser_enable = True

    # Disable the TDD engine while its configuration is updated.
    tdd.enable = False

    # Use an external trigger source to start the TDD sequence.
    tdd.sync_external = True

    # Begin generating triggers immediately after synchronisation.
    tdd.startup_delay_ms = 0

    # Calculate the Pulse Repetition Interval (PRI).
    #
    # The PRI consists of:
    #   - the FMCW ramp (chirp) duration
    #   - additional padding time between chirps
    #
    # ramp_time_us is stored in microseconds, so convert to milliseconds
    # before adding the padding value.
    PRI_ms = (config.ramp_time_us / 1e3) + config.pri_padding_ms

    
    # Set the time between successive chirp triggers.
    tdd.frame_length_ms = PRI_ms
    
    # Generate one trigger event per chirp in the burst.
    tdd.burst_count = config.num_chirps

    tdd.channel[0].enable = True
    tdd.channel[0].polarity = False
    tdd.channel[0].on_raw = config.tdd_trigger_on_raw
    tdd.channel[0].off_raw = config.tdd_trigger_off_raw

    tdd.channel[1].enable = True
    tdd.channel[1].polarity = False
    tdd.channel[1].on_raw = config.tdd_trigger_on_raw
    tdd.channel[1].off_raw = config.tdd_trigger_off_raw
    
    tdd.channel[2].enable = True
    tdd.channel[2].polarity = False
    tdd.channel[2].on_raw = config.tdd_trigger_on_raw
    tdd.channel[2].off_raw = config.tdd_trigger_off_raw
    
    # Apply the configuration and start the TDD engine.
    tdd.enable = True

    display(Markdown(f"""- Frame Len =  {tdd.frame_length_ms} ms
- No. Chirps / trigger =  {config.num_chirps}"""))

def tx_baseband():
    """
    Generate baseband transmit waveform (tone at IF)
    """ 
    fs = int(sdr.sample_rate)
    N = config.sdr_buf_size
    t = np.arange(N) / fs  # FIXED: Guarantees exactly N samples
    
    i_data = np.cos(2 * np.pi * t * config.signal_freq) * 2**14
    q_data = np.sin(2 * np.pi * t * config.signal_freq) * 2**14
    iq_data = i_data + 1j * q_data

    # Validate buffer length BEFORE sending to SDR
    expected_tx_buffer = sdr.tx_buffer_size
    actual_samples = len(iq_data)
    
    
    
    if actual_samples != expected_tx_buffer:
        raise ValueError(f"❌ Buffer length mismatch! Expected {expected_tx_buffer}, got {actual_samples}")
    
    # Send waveform to TX buffer
    sdr.tx([iq_data, iq_data])
    display(Markdown(f"""- Transmit {config.signal_freq/1e3} kHz
- TX buffer size: {expected_tx_buffer} (expected) vs {actual_samples} (actual)
- TX buffer loaded successfully"""))

def hardware_configuration(connect = True):
    """
    """
    if connect is True:
        display(Markdown("**Connect to Devices**"))
        connect_devices()
    
    display(Markdown("**Configure Phaser**"))
    configure_phaser()
    
    display(Markdown("**Configure PlutoSDR**"))
    destroy_sdr_buffer()
    configure_sdr()
    
    display(Markdown("**Configure ADF4159**"))
    configure_adf4159()
    
    display(Markdown("**Configure TDD Engine**"))
    configure_tdd()
    
    display(Markdown("**Transmit Baseband Signal**"))
    tx_baseband()

# FMCW RADAR: Target Detection and CFAR
## Adaptive Thresholding for Robust Detection

## Overview

In previous notebooks, we learned how to:
- Measure range (notebook 3)
- Measure velocity and generate Range-Doppler Maps(notebook 4)


We were able to see targets range and velocity, but the displays were a little noisey and cluttered.
This tutorial is going to look at tidying up (declutter) the Range-Doppler maps and attempt to identify targets using a technique know as **Constant False Alarm Rate** or **CFAR**


This notebook introduces **CFAR (Constant False Alarm Rate)** detection, an adaptive thresholding technique that maintains consistent detection performance across varying signal environments.

## Learning Objectives

By the end of this notebook, you will:
1. Understand **MTI**
1. Understand **detection theory** fundamentals (Pd vs Pfa)
2. Learn why **fixed thresholds fail** in real RADAR systems
3. Implement **CA-CFAR** (Cell-Averaging CFAR) for 1D and 2D data
4. Understand **OS-CFAR**, **GO-CFAR**, and **SO-CFAR** variants
5. Apply CFAR to **Range profiles** and **Range-Doppler Maps**
6. Analyze **detection performance** (ROC curves)
7. Handle **multi-target scenarios** and **clutter edges**
8. Implement **practical CFAR** for the CN0566 platform

# 1: Moving Target Indicator

To make it easy to identify moving targets on our range dopper plot, we can remove stationary objects.
There a number of variation of this theme that essentially removes statics signals.

Static targets will return a velocity of 0.

The simplest form of MTI implementation is to removed the average of the slow time signal bofore peforming the slow time fft:

As a reminder:

<div style="text-align: center;">
     <img src="resources/fast-slow-time.drawio.png" alt="fast and slow time" width="500">
</div>

Prior to taking the 2-dimensional FFT, we can process the slow time columns to remove the average or DC value.
Mathematically we're implementing the following:

$
\Large
\bar{x}[m] = \frac{1}{N} \sum_{n=0}^{N-1} x[n,m]
$

where:
* $n$ = chirp index (slow time)  
* $m$= sample index within a chirp (fast time)  

then:
* $N$ = number of chirps   
* $M$ = number of samples per chirp

And then simply subract the averarge.

$
\Large
 x_{MTI}[n,m] = x[n,m] - \bar{x}[m]
$

Upon subtracting the avergage, the 2D FFT can then be run to produce the bins requires for a range-dopper plot.


## MTI Demo 

The code is essentially identica the previous Range-Doppler demo, with the addition of a a basic implementation of MTI

In [ ]:
# Range-Doppler demo with MTI
# Run the previous demo cell once to define its helpers; pause it before configuring this demo.
# Reuse capture_data, combine_rx_channels, extract_chirps, process_chirps and render_plot.
# Run the preceding 128-chirp hardware configuration cell, then run this cell and press Play.
from IPython.display import HTML

# Detach the previous Play widget when this cell is run again.
if "play" in globals() and isinstance(play, widgets.Play):
    play.playing = False
    play.disabled = True
    play.unobserve_all(name="value")
if "rd_plots" in globals() and rd_plots is not None:
    plt.close(rd_plots["fig"])

# Parameters for this demo (other RadarConfig values remain as configured).
RD_MAX_RANGE_M = 15             # Apparent range: cable/path delay is not removed.
RD_PHASE_CORRECTION_DEG = 59.0   # Added per chirp interval; set to 0 for raw Doppler.
RD_RAW_CHIRPS_TO_SHOW = 2        # Display two chirps; process all 128.
RD_DYNAMIC_RANGE_DB = 60

def process_range_doppler(chirps, sample_rate, pri_s, 
                          ramp_s, bandwidth_hz,
                          if_hz, carrier_hz, 
                          phase_correction_deg, max_range_m):
    """
    Keep complex range FFT values so their slow-time phase contains velocity.
    
    """
    
    num_chirps, num_samples = chirps.shape
    wavelength = 3e8 / carrier_hz
    slope      = bandwidth_hz / ramp_s

    # 1. FAST TIME: FFT across columns (samples within each chirp).
    fast_window = np.blackman(num_samples)
    range_fft   = process_chirps(chirps, return_complex=True) / fast_window.sum()
    frequencies = np.fft.fftfreq(num_samples, d=1 / sample_rate)
    ranges      = (frequencies - if_hz) * 3e8 / (2 * slope)
    
    # Select the positive-range side of the IF, matching the previous display.
    visible    = (frequencies >= if_hz) & (ranges <= max_range_m)
    range_bins = ranges[visible]
    
    if len(range_bins) < 2:
        raise ValueError("Not enough range bins in the display interval; check sample rate and IF.")
    
    range_fft   = range_fft[:, visible]
    range_power = np.mean(np.abs(range_fft) ** 2, axis=0)
    range_db    = 10 * np.log10(range_power + 1e-12)

    # Apply the measured inter-chirp correction as a phase progression, not
    # a constant rotation of every row (which would not shift Doppler).
    correction = np.exp(1j * np.arange(num_chirps) * np.deg2rad(phase_correction_deg))
    range_fft = range_fft * correction[:, None]

    # Implement MTI

    # 2. SLOW TIME: FFT down rows (successive chirps at each range bin).
    # Do not take abs() before this FFT: it would discard Doppler phase.
    slow_window   = np.hanning(num_chirps)
    doppler_fft   = np.fft.fft(range_fft * slow_window[:, None], axis=0) / slow_window.sum()
    doppler_fft   = np.fft.fftshift(doppler_fft, axes=0)
    doppler_hz    = np.fft.fftshift(np.fft.fftfreq(num_chirps, d=pri_s))
    velocity_bins = doppler_hz * wavelength / 2
    map_db        = 20 * np.log10(np.abs(doppler_fft) + 1e-12)
    map_db       -= map_db.max()  # Colour is dB relative to the strongest bin in this CPI.
    
    return range_bins, velocity_bins, range_db, map_db

def create_rd_plots(sample_rate, pri_s, ramp_s, 
                    skip_fraction, range_bins, 
                    velocity_bins):
    """
    Create a raw-buffer plot, averaged range FFT, and range-Doppler map.
    """
    fig = Figure(figsize=(14, 15), dpi=90)
    canvas = FigureCanvasAgg(fig)
    
    gs = fig.add_gridspec(2, 2, height_ratios=[1, 2])
    ax_time = fig.add_subplot(gs[0, 0])
    ax_range = fig.add_subplot(gs[0, 1])
    ax_rd = fig.add_subplot(gs[1, :])
   
    line_i, = ax_time.plot([], [], linewidth=0.6, alpha=0.7, label='I (RX0 + RX1)')
    line_q, = ax_time.plot([], [], linewidth=0.6, alpha=0.7, label='Q (RX0 + RX1)')
    line_abs, = ax_time.plot([], [], color='black', linewidth=0.9, label='Magnitude')
    
    for chirp in range(min(RD_RAW_CHIRPS_TO_SHOW, config.num_chirps)):
        start_us = chirp * pri_s * 1e6
        end_us = start_us + ramp_s * 1e6
        ax_time.axvline(start_us, color='tab:green', linestyle='--',
                        label='Expected ramp start' if chirp == 0 else None)
        ax_time.axvline(end_us, color='tab:red', linestyle='--',
                        label='Expected ramp end' if chirp == 0 else None)
        ax_time.axvspan(start_us, start_us + skip_fraction * ramp_s * 1e6,
                        color='gold', alpha=0.2, label='Excluded settling' if chirp == 0 else None)
        ax_time.axvspan(end_us, start_us + pri_s * 1e6, color='grey', alpha=0.15,
                        label='Padding' if chirp == 0 else None)
        
    ax_time.set_xlabel('Time from first RX sample (us)')
    ax_time.set_ylabel('Raw amplitude (ADC counts)')
    ax_time.set_xlim(0, min(RD_RAW_CHIRPS_TO_SHOW, config.num_chirps) * pri_s * 1e6)
    ax_time.set_title('Raw dechirped I/Q - expected first trigger at t = 0')
    ax_time.legend(loc='lower left', ncol=4, fontsize=8)
    ax_time.grid(True, alpha=0.3)

    line_range, = ax_range.plot([], [], color='tab:blue', linewidth=1.2)
    ax_range.set(xlabel='Apparent range (m)', ylabel='Mean power (dB re 1 ADC count squared)',
                 xlim=(0, RD_MAX_RANGE_M), title=f'Fast-time FFT - power averaged over {config.num_chirps} chirps')
    ax_range.grid(True, alpha=0.3)

    # imshow takes pixel edges; the FFT arrays contain pixel centres.
    dr = range_bins[1] - range_bins[0]
    dv = velocity_bins[1] - velocity_bins[0]
    
    extent = [velocity_bins[0] - dv/2, velocity_bins[-1] + dv/2,
              range_bins[0] - dr/2, range_bins[-1] + dr/2]
   
    image = ax_rd.imshow(np.full((len(range_bins), len(velocity_bins)), -RD_DYNAMIC_RANGE_DB),
                         origin='lower', aspect='auto', interpolation='nearest', extent=extent,
                         cmap='inferno', vmin=-RD_DYNAMIC_RANGE_DB, vmax=0)
    
    ax_rd.set(xlabel='Radial velocity (m/s)', ylabel='Apparent range (m)',
              xlim=(-5, 5), ylim=(0, RD_MAX_RANGE_M), title='Range-Doppler map - fast-time then slow-time FFT')
    
    ax_rd.axvline(0, color='white', linewidth=0.7, alpha=0.5)
    
    
    fig.tight_layout()
    
    return dict(fig=fig, canvas=canvas, ax_time=ax_time, ax_range=ax_range, ax_rd=ax_rd,
                line_i=line_i, line_q=line_q, line_abs=line_abs, line_range=line_range,
                image=image)


def update_rd_plots(plots, rx_data, sample_rate, 
                    pri_s, range_bins, range_db, 
                    map_db, frame_count):
    """
    Update the same three plots for each captured CPI.
    """
    shown = min(len(rx_data), round(min(RD_RAW_CHIRPS_TO_SHOW, config.num_chirps) * pri_s * sample_rate))
    raw = rx_data[:shown]
    
    times_us = np.arange(shown) / sample_rate * 1e6
    
    plots['line_i'].set_data(times_us, raw.real)
    plots['line_q'].set_data(times_us, raw.imag)
    plots['line_abs'].set_data(times_us, np.abs(raw))
   
    limit = max(float(np.abs(raw).max()), 1.0)
    
    plots['ax_time'].set_ylim(-1.1 * limit, 1.1 * limit)
    plots['ax_time'].set_title(
        f'Raw I/Q - first {min(RD_RAW_CHIRPS_TO_SHOW, config.num_chirps)} of {config.num_chirps} chirps '
        f'- frame {frame_count} (assuming trigger at t = 0)')
    
    plots['line_range'].set_data(range_bins, range_db)
    plots['ax_range'].set_ylim(range_db.max() - RD_DYNAMIC_RANGE_DB, range_db.max() + 5)
    plots['ax_rd'].set_title(f'Range-Doppler map - {config.num_chirps} chirps - frame {frame_count}')
    plots['image'].set_data(map_db.T)  # Rows are range; columns are velocity.

# Hardware Configuration
hardware_configuration()

# Read timing once, after running the hardware configuration cell above.
rd_sample_rate  = float(sdr.sample_rate)
rd_pri_s        = float(tdd.frame_length_ms) * 1e-3
rd_ramp_s       = float(phaser.freq_dev_time) * 1e-6
rd_bandwidth_hz = 4 * float(phaser.freq_dev_range)

if config.num_chirps < 3 or int(tdd.burst_count) != config.num_chirps:
    raise ValueError("Run the preceding range-Doppler hardware configuration cell first.")

if not 0 <= config.begin_offset_fraction < 1:
    raise ValueError("Settling fraction must be between 0 and 1.")

rd_skip_samples = int(config.begin_offset_fraction * config.ramp_time_us * 1e-6 * rd_sample_rate)

if int(config.ramp_time_us * 1e-6 * rd_sample_rate) - rd_skip_samples < 3:
    raise ValueError("Too few usable samples per ramp for the range FFT.")

# One layout, one image widget, and one callback for each Play value change.
out_info = widgets.Output(
    layout=widgets.Layout(border='1px solid #ccc', padding='10px'))

plot_widget = widgets.Image(
    format='png', layout=widgets.Layout(width='100%', border='1px solid #ccc'))

play = widgets.Play(value=0, min=0, max=1000, step=1, interval=100, description="Press play", align = "centre")

rd_plots        = None
rd_frame_count  = 0
rd_needs_warmup = True


def update(change):
    """Capture one CPI, process it, and refresh the displayed image."""
    global rd_plots, rd_frame_count, rd_needs_warmup
    
    if play.disabled or change['new'] <= change['old']:
        return  # Reset/rewind does not acquire a frame.
    try:
        if rd_needs_warmup:
            capture_data(phaser, sdr)  # Discard the first capture once, on first Play.
            rd_needs_warmup = False
        raw_data = capture_data(phaser, sdr)
        rx_data = combine_rx_channels(raw_data)
        chirps = extract_chirps(rx_data, config.num_chirps, rd_sample_rate, config)
        chirps = chirps[:, rd_skip_samples:]
        ranges, velocities, range_db, map_db = process_range_doppler(
            chirps, rd_sample_rate, rd_pri_s, rd_ramp_s, rd_bandwidth_hz,
            config.signal_freq, config.output_freq, RD_PHASE_CORRECTION_DEG, RD_MAX_RANGE_M)
        if rd_plots is None:
            rd_plots = create_rd_plots(rd_sample_rate, rd_pri_s, rd_ramp_s,
                                       config.begin_offset_fraction, ranges, velocities)
        rd_frame_count += 1
        update_rd_plots(rd_plots, rx_data, rd_sample_rate, rd_pri_s,
                        ranges, range_db, map_db, rd_frame_count)
        plot_widget.value = render_plot(rd_plots['canvas'])

    except (Exception, KeyboardInterrupt) as error:
        # Callback errors must be shown in the Output widget, not lost in kernel logs.
        play.playing = False
        play.disabled = True
        with out_info:
            clear_output(wait=True)
            print(f'Capture stopped: {type(error).__name__}: {error}\n'
                                   'Fix the problem, then rerun this cell to enable Play again.')


play.observe(update, names='value')

layout = widgets.VBox([out_info, play, plot_widget])

# Setup information is displayed once; only the plot changes on each frame.
with out_info:
    clear_output(wait=True)
    display(HTML(
        f'<b>Range-Doppler setup &middot; {config.num_chirps} chirps per capture</b><br>'
        f'Sample rate: {rd_sample_rate/1e6:.3f} MS/s &middot; '
        f'RX buffer: {sdr.rx_buffer_size:,} samples<br>'
        f'Ramp: {rd_ramp_s * 1e6:.0f} us &middot; PRI: {rd_pri_s * 1e6:.0f} us '
        f'&middot; CPI: {config.num_chirps * rd_pri_s * 1e3:.1f} ms<br>'
        f'Doppler bin: {(3e8 / config.output_freq) / (2 * config.num_chirps * rd_pri_s):.3f} m/s '
        f'&middot; phase correction: {RD_PHASE_CORRECTION_DEG:+.1f}&deg;/chirp<br>'
        f'Range display: 0&ndash;{RD_MAX_RANGE_M:g} m (includes cable/path delay). '
        'Press Play to start; Pause to stop updates.'
    ))

display(layout)


## Section 6: Target Detection in Range-Doppler Space

### 2D Peak Detection

To extract target parameters:
1. Apply threshold to RDM
2. Find local maxima (peaks)
3. For each peak:
   - Range bin → range
   - Doppler bin → velocity
   - Peak magnitude → RCS/SNR

TODO: Implement 2D peak detection

## Section 7: MTI (Moving Target Indication) Filtering

### The Static Clutter Problem

In many scenarios, **static clutter** (walls, ground, stationary objects) dominates the RDM at **zero Doppler**, masking weak moving targets.

### MTI: High-Pass Filter in Slow-Time

MTI removes the zero-Doppler component by:
1. Subtracting mean across chirps (simple MTI)
2. Or applying high-pass filter along slow-time

$$
\text{MTI: } x_{\text{MTI}}[m, n] = x[m, n] - \frac{1}{M}\sum_{m=0}^{M-1} x[m, n]
$$

This **removes DC** in the slow-time domain → removes static clutter.

### Higher-Order MTI

For better clutter rejection:
- **2-pulse canceller**: $y[m] = x[m] - x[m-1]$
- **3-pulse canceller**: $y[m] = x[m] - 2x[m-1] + x[m-2]$

TODO: Implement MTI filtering

In [ ]:
# TODO: MTI filter
def apply_mti(data_cube, order=1):
    """
    Apply Moving Target Indication filter
    
    Parameters:
    -----------
    data_cube : ndarray
        Shape (M, N, channels)
    order : int
        MTI filter order (1=simple mean removal, 2=2-pulse canceller)
    
    Returns:
    --------
    data_mti : ndarray
        Clutter-suppressed data
    """
    # TODO: Implement
    # - Subtract mean across chirps (axis 0)
    # - Or apply differencing filter
    pass

## Section 1: Detection Theory Fundamentals

### The Detection Problem

RADAR detection is a **binary hypothesis test**:

- **H₀ (null hypothesis)**: No target present → signal = noise only
- **H₁ (alternative hypothesis)**: Target present → signal = target + noise

We set a **threshold** $T$ and declare:
- Detection if: $|x| > T$
- No detection if: $|x| \leq T$

### Four Possible Outcomes

| **Truth** | **Decision** | **Outcome** | **Notation** |
|-----------|--------------|-------------|-------------|
| Target present (H₁) | Detect | ✓ **Correct Detection** | $P_d$ (Detection Probability) |
| Target present (H₁) | No detect | ✗ **Miss** | $P_m = 1 - P_d$ (Miss Probability) |
| No target (H₀) | Detect | ✗ **False Alarm** | $P_{fa}$ (False Alarm Probability) |
| No target (H₀) | No detect | ✓ **Correct Rejection** | $1 - P_{fa}$ |

### The Threshold Trade-off

- **Lower threshold** → More detections BUT more false alarms
- **Higher threshold** → Fewer false alarms BUT more misses

```
Noise PDF    Target+Noise PDF
    |            |
    |      ___   |    ___
    |    /     \ |  /     \
    |   /       \| /       \
    |  /         X         \
    | /       / | \         \
    |/      /   |  \         \
   _|_____/____|___\__________\_____
            Threshold T
            
   |<- Pfa ->|    |<-- Pd -->|
```

### Receiver Operating Characteristic (ROC) Curve

A plot of $P_d$ vs $P_{fa}$ as the threshold varies:

- **Ideal detector**: $P_d = 1$, $P_{fa} = 0$ (top-left corner)
- **Random guessing**: $P_d = P_{fa}$ (diagonal line)
- **Better detectors**: ROC curve closer to top-left

Typical RADAR specs: $P_d = 0.9$ (90%) at $P_{fa} = 10^{-6}$ (one false alarm per million cells)

## Section 2: Why Fixed Thresholds Fail

### Problem 1: Spatially Varying Noise

In real RADAR:
- **Thermal noise** varies with receiver gain and temperature
- **Clutter** (ground, sea, weather) has different RCS in different regions
- **Interference** appears at specific ranges/angles

**Result**: A fixed threshold set for one region gives:
- Too many false alarms in high-noise regions
- Missed detections in low-noise regions

### Problem 2: Non-Stationary Environment

Noise statistics change over time:
- Weather moves through the scene
- Interference appears and disappears
- Receiver gain adjusts (AGC)

A threshold set at $t=0$ becomes invalid at $t=10$ seconds.

### Problem 3: Clutter Edges

At transitions between clutter types (e.g., land → sea):
- Sharp change in background power level
- Fixed threshold either:
  - Misses targets in low-clutter region
  - False alarms in high-clutter region

### The Solution: Adaptive Thresholding

Set the threshold **locally** based on the **local noise statistics**:

$$
T[i] = \alpha \cdot \hat{\sigma}_i^2
$$

Where:
- $\hat{\sigma}_i^2$ = estimated noise power **around cell $i$**
- $\alpha$ = threshold factor (controls $P_{fa}$)

This is the **CFAR** approach.

## Section 3: CA-CFAR (Cell-Averaging CFAR)

### The CA-CFAR Algorithm

For each cell under test (CUT), estimate the local noise power by **averaging nearby cells**.

```
Range bins →
┌─┬─┬─┬─┬─┬─┬─┬─┬─┬─┬─┬─┬─┬─┬─┐
│ Training │Guard│CUT│Guard│ Training │
│  Cells   │Cells│   │Cells│  Cells   │
└─┴─┴─┴─┴─┴─┴─┴─┴─┴─┴─┴─┴─┴─┴─┘
  ↑                             ↑
  N_train cells         N_train cells
         ↑         ↑
      N_guard   N_guard
```

### Components

1. **CUT (Cell Under Test)**: The cell being tested for target presence
2. **Guard Cells**: Cells immediately adjacent to CUT (excluded from averaging)
   - Purpose: Prevent target energy from leaking into noise estimate
   - Typical: 1-4 cells on each side
3. **Training Cells**: Cells used to estimate noise level
   - Purpose: Provide noise statistics
   - Typical: 10-50 cells on each side

### CA-CFAR Formula

$$
Z = \frac{1}{N_{\text{train}}} \sum_{\text{training cells}} |x_i|^2
$$

$$
T = \alpha \cdot Z
$$

$$
\text{Detection} = 
\begin{cases}
1 & \text{if } |x_{\text{CUT}}|^2 > T \\
0 & \text{otherwise}
\end{cases}
$$

### Choosing $\alpha$ (Threshold Factor)

For Rayleigh-distributed noise (typical for RADAR):

$$
P_{fa} = e^{-\alpha}
$$

So for $P_{fa} = 10^{-6}$:

$$
\alpha = -\ln(P_{fa}) = -\ln(10^{-6}) \approx 13.8
$$

In **dB**: $\alpha_{\text{dB}} = 10\log_{10}(\alpha) \approx 11.4$ dB

### Guard Cell Sizing

Number of guard cells depends on **target width in bins**:

For range:
$$
N_{\text{guard}} \geq \frac{\text{Target extent}}{\Delta R}
$$

Typical: 2-4 cells per side

## Section 4: Implementing CA-CFAR

### Setup and Imports

TODO: Import libraries

In [ ]:
# TODO: Imports
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.stats import chi2, rayleigh
import ipywidgets as widgets
from IPython.display import display

# TODO: Add phaser-specific imports if using hardware
# from adi import ad9361
# from adi.cn0566 import CN0566

### 1D CA-CFAR Implementation

TODO: Implement 1D CA-CFAR for range profiles

In [ ]:
# TODO: 1D CA-CFAR function
def ca_cfar_1d(data, num_train, num_guard, pfa=1e-6, method='average'):
    """
    1D Cell-Averaging CFAR detector
    
    Parameters:
    -----------
    data : ndarray
        Input signal (range profile), power values
    num_train : int
        Number of training cells on each side
    num_guard : int
        Number of guard cells on each side
    pfa : float
        Desired false alarm probability
    method : str
        'average' (CA-CFAR), 'greatest_of' (GO-CFAR), 'smallest_of' (SO-CFAR)
    
    Returns:
    --------
    detections : ndarray (bool)
        Detection mask (True = target detected)
    threshold : ndarray
        Adaptive threshold for each cell
    """
    N = len(data)
    detections = np.zeros(N, dtype=bool)
    threshold = np.zeros(N)
    
    # Calculate threshold factor alpha from Pfa
    alpha = -np.log(pfa)
    
    # Process each cell
    for i in range(N):
        # TODO: Implement
        # 1. Define training cell indices (exclude CUT and guard cells)
        # 2. Extract training cells (handle edges)
        # 3. Compute noise estimate Z
        # 4. Compute threshold T = alpha * Z
        # 5. Compare data[i] to threshold
        pass
    
    return detections, threshold

### Testing CA-CFAR with Synthetic Data

TODO: Create synthetic range profile with targets and noise

In [ ]:
# TODO: Synthetic data generation
def generate_test_signal(N=512, target_snr_db=15, noise_variance=1.0):
    """
    Generate synthetic range profile with targets
    
    Returns:
    --------
    signal : complex noise + targets
    target_indices : ground truth target locations
    """
    # TODO: Implement
    # 1. Generate complex Gaussian noise
    # 2. Add targets at specific ranges with known SNR
    # 3. Optionally add clutter region
    pass

### Visualization: Fixed Threshold vs CA-CFAR

TODO: Side-by-side comparison

In [ ]:
# TODO: Comparison demo
# - Generate test signal
# - Apply fixed threshold
# - Apply CA-CFAR
# - Plot results side-by-side
# - Count false alarms and missed detections

## Section 5: CFAR Variants

### GO-CFAR (Greatest-Of CFAR)

**Problem with CA-CFAR**: Fails at clutter edges

At a transition from low to high clutter:
- Leading window: low clutter
- Trailing window: high clutter
- Average is **between** the two → threshold too low in high clutter

**GO-CFAR Solution**: Use the **greater** of the two averages

$$
Z_{\text{leading}} = \frac{1}{N} \sum_{\text{leading window}} |x_i|^2
$$

$$
Z_{\text{lagging}} = \frac{1}{N} \sum_{\text{lagging window}} |x_i|^2
$$

$$
Z_{\text{GO}} = \max(Z_{\text{leading}}, Z_{\text{lagging}})
$$

**Trade-off**: Higher $P_{fa}$ in homogeneous regions (more conservative)

### SO-CFAR (Smallest-Of CFAR)

Use the **smaller** of the two averages:

$$
Z_{\text{SO}} = \min(Z_{\text{leading}}, Z_{\text{lagging}})
$$

**Use case**: When you want aggressive detection (prefer false alarms over misses)

### OS-CFAR (Ordered-Statistics CFAR)

**Problem**: Interfering targets in training cells bias the noise estimate

**OS-CFAR Solution**: Sort training cells and use a **rank-order statistic** (e.g., median)

Algorithm:
1. Collect training cell values: $\{x_1, x_2, \ldots, x_N\}$
2. Sort in ascending order: $x_{(1)} \leq x_{(2)} \leq \cdots \leq x_{(N)}$
3. Select $k$-th order statistic: $Z = x_{(k)}$
4. Threshold: $T = \alpha \cdot Z$

Common choices:
- $k = 3N/4$ (75th percentile) → robust to interferers
- $k = N/2$ (median) → very robust, but higher loss

**Advantage**: Robust to outliers (interfering targets don't bias threshold)

**Disadvantage**: Computationally more expensive (sorting)

## Section 6: 2D CFAR for Range-Doppler Maps

### Extending CFAR to 2D

Range-Doppler Maps are 2D: need to estimate noise in **both dimensions**.

```
Doppler →
┌─────────────────────────────┐  ↑
│         Training            │  │
│  ┌──────────────────────┐   │  │
│  │      Guard           │   │  │
│  │   ┌──────────────┐   │   │  Range
│  │   │    CUT       │   │   │  │
│  │   └──────────────┘   │   │  │
│  │                      │   │  │
│  └──────────────────────┘   │  │
│                             │  ↓
└─────────────────────────────┘
```

### 2D CA-CFAR Algorithm

For a CUT at position $(i, j)$:

1. Define rectangular **training region** around CUT
2. Exclude **guard region** (rectangle around CUT)
3. Average all training cells
4. Apply threshold

$$
Z_{i,j} = \frac{1}{N_{\text{train}}} \sum_{(m,n) \in \text{training}} |X[m,n]|^2
$$

$$
T_{i,j} = \alpha \cdot Z_{i,j}
$$

$$
\text{Detection}_{i,j} = |X[i,j]|^2 > T_{i,j}
$$

### Implementation Approaches

**Naive approach**: Nested loops (slow)

**Fast approach**: Use 2D convolution
1. Create training cell mask (1s in training region, 0s elsewhere)
2. Convolve mask with RDM → gives sum of training cells
3. Divide by number of training cells → noise estimate

TODO: Implement 2D CA-CFAR

In [ ]:
# TODO: 2D CA-CFAR function
def ca_cfar_2d(rdm, num_train_range, num_train_doppler, 
               num_guard_range, num_guard_doppler, pfa=1e-6):
    """
    2D Cell-Averaging CFAR for Range-Doppler Maps
    
    Parameters:
    -----------
    rdm : ndarray (M, N)
        Range-Doppler map (power)
    num_train_range : int
        Training cells in range dimension
    num_train_doppler : int
        Training cells in Doppler dimension
    num_guard_range : int
        Guard cells in range dimension
    num_guard_doppler : int
        Guard cells in Doppler dimension
    pfa : float
        False alarm probability
    
    Returns:
    --------
    detections : ndarray (bool)
        Detection mask
    threshold : ndarray
        Adaptive threshold map
    """
    # TODO: Implement
    # Option 1: Nested loops (simple but slow)
    # Option 2: Convolution-based (fast)
    pass

### 2D CFAR Demo with Synthetic RDM

TODO: Generate synthetic Range-Doppler map and apply 2D CFAR

In [ ]:
# TODO: 2D CFAR demo
# - Generate RDM with multiple targets at different SNRs
# - Add spatially-varying noise
# - Apply 2D CFAR
# - Visualize: RDM, threshold surface, detections

## Section 7: Performance Analysis

### Monte Carlo Simulation for ROC Curves

To characterize detector performance:

1. Generate many realizations of noise + target
2. For each threshold value, count:
   - True detections (when target present)
   - False alarms (when target absent)
3. Compute $P_d$ and $P_{fa}$
4. Plot ROC curve

TODO: Implement ROC analysis

In [ ]:
# TODO: ROC curve generator
def generate_roc_curve(detector_func, num_trials=1000, snr_db=10):
    """
    Generate ROC curve for a detector via Monte Carlo
    
    Parameters:
    -----------
    detector_func : function
        Detector function (fixed threshold or CFAR)
    num_trials : int
        Number of Monte Carlo trials
    snr_db : float
        Target SNR in dB
    
    Returns:
    --------
    pfa_list : list
        False alarm probabilities
    pd_list : list
        Detection probabilities
    """
    # TODO: Implement Monte Carlo loop
    pass

### Comparing CFAR Variants

TODO: Performance comparison across CA-CFAR, GO-CFAR, OS-CFAR

In [ ]:
# TODO: CFAR variant comparison
# - Test all variants on same data
# - Scenarios:
#   * Homogeneous noise
#   * Clutter edge
#   * Multiple targets
#   * Interfering targets in training cells
# - Plot ROC curves
# - Show detection maps

### CFAR Loss

**CFAR Loss**: The additional SNR required (compared to optimal detector) to achieve the same $P_d$ at a given $P_{fa}$.

Typical values:
- CA-CFAR: ~0.5-1 dB loss
- GO-CFAR: ~1-2 dB loss (more conservative)
- OS-CFAR: ~1-3 dB loss (depends on rank)

**Trade-off**: CFAR provides robustness at the cost of slightly worse detection performance in ideal (homogeneous) conditions.

## Section 8: Practical Considerations

### Choosing CFAR Parameters

**Training cells ($N_{\text{train}}$)**:
- Too few: Poor noise estimate (high variance)
- Too many: Slow adaptation, can't track spatial variations
- **Rule of thumb**: 10-50 cells per side

**Guard cells ($N_{\text{guard}}$)**:
- Must be ≥ target extent in bins
- Too few: Target leaks into noise estimate → threshold too high → miss
- Too many: Wasted cells, slower spatial adaptation
- **Rule of thumb**: 2-4 cells per side

**False alarm rate ($P_{fa}$)**:
- Typical: $10^{-4}$ to $10^{-6}$
- Lower $P_{fa}$ → higher threshold → more misses
- Application-dependent:
  - Safety-critical (automotive): prefer false alarms over misses
  - Surveillance: balance based on human operator load

### Edge Handling

At the edges of the data (first/last bins):

**Option 1**: Reduce window size (asymmetric window)
**Option 2**: Pad with median value
**Option 3**: Declare edges as "no decision" zone

### Computational Complexity

For 1D signal of length $N$:
- **CA-CFAR**: $O(N \cdot N_{\text{train}})$ (can be optimized to $O(N)$ with running sum)
- **OS-CFAR**: $O(N \cdot N_{\text{train}} \log N_{\text{train}})$ (sorting)

For 2D RDM of size $M \times N$:
- Naive: $O(MN \cdot W^2)$ where $W$ is window size
- Optimized (convolution): $O(MN \log(MN))$ using FFT

**Real-time considerations**: For high frame rates, use optimized implementations or GPU acceleration

## Section 9: CFAR on Real CN0566 Data

### Hardware Setup

TODO: Connect to CN0566 and configure for FMCW

In [ ]:
# TODO: Hardware setup
# - Connect to CN0566
# - Configure FMCW chirp (500 MHz BW, TDD)
# - Set up beamformer (boresight)

### Applying CFAR to Range Profiles

TODO: Capture range profile and apply 1D CFAR

In [ ]:
# TODO: 1D CFAR on range profile
# - Capture single chirp
# - Compute range FFT
# - Apply CA-CFAR
# - Plot: range profile, threshold, detections
# - Report detected ranges

### Applying CFAR to Range-Doppler Maps

TODO: Capture CPI and apply 2D CFAR

In [ ]:
# TODO: 2D CFAR on RDM
# - Capture CPI (64 chirps)
# - Compute 2D FFT (Range-Doppler)
# - Apply 2D CA-CFAR
# - Overlay detections on RDM
# - Extract target list: (range, velocity, SNR)

### Interactive CFAR Demo

TODO: Live updating display with adjustable CFAR parameters

In [ ]:
# TODO: Interactive CFAR demonstration
# Widgets for:
# - Pfa slider (10^-3 to 10^-7)
# - Num train cells slider
# - Num guard cells slider
# - CFAR variant selection (CA, GO, SO, OS)
# 
# Real-time display:
# - RDM with threshold overlay
# - Detected targets marked
# - Target list table
# - False alarm count

## Section 10: Multi-Target Scenarios

### Closely Spaced Targets

**Problem**: When two targets are close in range/Doppler:
- Their responses overlap (sidelobe interaction)
- CFAR may detect as single target
- Or miss the weaker target

**Solution**: 
- Use narrower guard cells
- Apply **peak finding** after CFAR (find local maxima)
- Consider super-resolution techniques (MUSIC, ESPRIT)

### Masking Effect

Strong target can **mask** nearby weak targets:
1. Strong target detected
2. Its sidelobes raise noise estimate in surrounding cells
3. Weak nearby target falls below locally elevated threshold

**Mitigation**:
- Use **censored** CFAR variants (OS-CFAR)
- Two-pass detection:
  1. Detect and remove strong targets
  2. Re-run CFAR on residual

TODO: Implement two-pass CFAR

In [ ]:
# TODO: Two-pass CFAR
def two_pass_cfar(data, cfar_params):
    """
    Two-pass CFAR for detecting weak targets near strong ones
    
    Pass 1: Detect strong targets
    Pass 2: Subtract strong targets, detect weak ones
    """
    # TODO: Implement
    pass

## Section 11: Integration with Tracking

### Detection to Track Pipeline

CFAR output feeds into a **tracker**:

```
Raw Data → 2D FFT → CFAR Detector → Peak Extraction
                                          ↓
                                    Detections
                                    (range, vel, SNR)
                                          ↓
                                  Data Association
                                    (match to tracks)
                                          ↓
                                   Kalman Filter
                                 (state estimation)
                                          ↓
                                  Track Maintenance
                                (create/delete tracks)
```

### Detection Quality Metrics

Beyond binary detection, pass quality metrics to tracker:
- **SNR**: Signal-to-Noise Ratio
- **Excess above threshold**: How far above threshold?
- **Peak width**: Narrow peak = point target, wide = distributed

Tracker uses these for:
- **Measurement covariance**: High SNR → low uncertainty
- **Gating**: Reject low-quality detections in clutter
- **Track scoring**: Promote tracks with consistent high-SNR detections

TODO: Implement detection quality scoring

In [ ]:
# TODO: Detection quality metrics
def compute_detection_quality(rdm, detections, threshold):
    """
    Compute quality metrics for each detection
    
    Returns:
    --------
    detections_with_quality : list of dict
        Each detection includes:
        - range_idx, doppler_idx
        - range_m, velocity_ms
        - snr_db
        - excess_db (above threshold)
        - peak_quality (0-1)
    """
    # TODO: Implement
    pass

## Section 12: Summary and Best Practices

### What We've Learned

In this notebook, we covered:

1. ✓ **Detection theory**: $P_d$, $P_{fa}$, ROC curves
2. ✓ **Why fixed thresholds fail** in real-world RADAR
3. ✓ **CA-CFAR algorithm** and implementation (1D and 2D)
4. ✓ **CFAR variants**: GO-CFAR, SO-CFAR, OS-CFAR
5. ✓ **Performance analysis**: ROC curves, CFAR loss
6. ✓ **Practical considerations**: Parameter selection, edge handling
7. ✓ **Real hardware application** on CN0566
8. ✓ **Multi-target scenarios** and mitigation strategies

### When to Use Which CFAR Variant

| Scenario | Recommended CFAR | Why |
|----------|------------------|-----|
| **Homogeneous clutter** | CA-CFAR | Simple, low loss |
| **Clutter edges** | GO-CFAR | Adapts to transitions |
| **Multiple targets** | OS-CFAR | Robust to interferers |
| **Need aggressive detection** | SO-CFAR | Low threshold |
| **Low complexity required** | CA-CFAR | Fastest |

### Design Guidelines

**Step 1**: Choose $P_{fa}$ based on application
- Safety-critical: $10^{-4}$ (1 false alarm per 10,000 cells)
- Surveillance: $10^{-6}$ (1 per million)

**Step 2**: Set guard cells = target extent
- Range: 2-4 cells (depends on chirp BW)
- Doppler: 1-2 cells (depends on CPI length)

**Step 3**: Set training cells for good statistics
- Start with 10-20 per side
- Increase if noise estimate is noisy
- Decrease if scene has fine spatial structure

**Step 4**: Test with synthetic data first
- Verify $P_{fa}$ is correct
- Measure $P_d$ vs SNR
- Test edge cases (clutter, interferers)

**Step 5**: Tune on real data
- Iterate parameters based on false alarm rate
- Validate detection performance

### Common Pitfalls

❌ **Too few training cells** → noisy threshold → erratic detections

❌ **Too few guard cells** → target energy biases threshold → missed detections

❌ **Wrong $P_{fa}$** → either swamped with false alarms or missing targets

❌ **Ignoring edges** → spurious detections or misses at boundaries

❌ **Not validating** → assuming CFAR "just works" without testing

### Next Steps

After mastering CFAR detection:
- **Notebook 6**: Integrate CFAR with beamforming for 3D tracking
- Implement **multi-target tracking** (Kalman filters, data association)
- Explore **machine learning** detection (CNN-based detectors)
- Study **non-Gaussian clutter** models (K-distribution, Weibull)

## Appendix A: Mathematical Derivations

### Threshold Factor for CA-CFAR

For Rayleigh-distributed magnitude noise, the power follows an **exponential distribution**:

$$
p(x) = \frac{1}{\sigma^2} e^{-x/\sigma^2}, \quad x \geq 0
$$

The noise estimate $Z$ is the average of $N$ exponential variables → scaled chi-squared distribution.

The false alarm probability is:

$$
P_{fa} = P(X > \alpha Z | H_0) = e^{-\alpha}
$$

Solving for $\alpha$:

$$
\alpha = -\ln(P_{fa})
$$

For $P_{fa} = 10^{-6}$:

$$
\alpha = -\ln(10^{-6}) = 13.816
$$

In dB: $10\log_{10}(13.816) \approx 11.4$ dB

## Appendix B: Optimized Implementations

### Fast 1D CA-CFAR with Running Sum

Instead of recomputing the sum for each cell, use a **sliding window**:

```python
# Precompute cumulative sum
cumsum = np.cumsum(data)

# For cell i, training sum is:
# cumsum[i - guard - 1] - cumsum[i - guard - train - 1]  (left)
# + cumsum[i + guard + train] - cumsum[i + guard]  (right)
```

This reduces complexity from $O(N \cdot N_{train})$ to $O(N)$.

### GPU-Accelerated 2D CFAR

For real-time processing, use GPU:
- Each CUT processed by one thread
- Shared memory for training cells
- Can process full RDM in <1 ms

Libraries: CuPy, Numba CUDA

## Appendix C: Further Reading

### Papers

- Finn & Johnson (1968): "Adaptive Detection Mode with Threshold Control as a Function of Spatially Sampled Clutter-Level Estimates"
- Rohling (1983): "Radar CFAR Thresholding in Clutter and Multiple Target Situations"
- Gandhi & Kassam (1988): "Analysis of CFAR Processors in Nonhomogeneous Background"

### Books

- *Detection, Estimation, and Modulation Theory* by Van Trees
- *Radar Systems Analysis and Design Using MATLAB* by Mahafza
- *Principles of Modern Radar: Advanced Techniques* by Richards et al.

### Standards

- IEEE Std 686: "Radar Signal Processing"
- ITU-R M.1464: "Characteristics of Radiolocation Radars"